# 🎮 ELYSIA — Conversational Video Game Recommendation Assistant
### Eksperimen & Prototyping Chatbot LLM Berbasis Google Gemini API & Playstyle DNA

Notebook ini merupakan tahap eksplorasi dan pembuatan prototipe interaktif untuk **ELYSIA** (*Emotionally-Adjusted Ludic Yield Spatial Integrated Assistant*). 

Dalam notebook ini, kita akan mempelajari dan menguji setiap komponen secara bertahap:
1. **Instalasi & Konfigurasi Lingkungan (API Key & .env)**
2. **Inisialisasi Client Google Gemini API**
3. **Perancangan Persona & System Prompt**
4. **Manajemen Riwayat Percakapan (Conversation History)**
5. **Streaming Response (Generasi Respons Real-Time)**
6. **Ekstraksi Preferensi & Mood Pengguna Berbasis LLM**
7. **Memuat Dataset & Prekomputasi Playstyle DNA Game (24.082 Game)**
8. **Algoritma Hybrid Recommendation (Playstyle DNA, Genre, Rating, Steam IDR)**
9. **Integrasi End-to-End (Percakapan Cerdas + Rekomendasi Game)**
10. **Menyimpan & Memuat Riwayat Percakapan (JSON)**
11. **Chatbot Loop Interaktif Lengkap di Notebook**


## 1. Pemasangan & Import Library

Kita menggunakan pustaka resmi:
* `google-generativeai`: SDK resmi untuk berinteraksi dengan model Gemini.
* `python-dotenv`: Membaca variabel rahasia (`.env`) secara aman.
* `pandas` & `numpy`: Pengolahan data game, vektorisasi, dan perhitungan jarak Euclidean DNA.


In [ ]:
# Pemasangan pustaka (uncomment jika belum terpasang di environment)
# !pip install -q google-generativeai python-dotenv pandas numpy

import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import google.generativeai as genai

print("✅ Library berhasil diimpor!")


## 2. Memuat Gemini API Key Secara Aman

Kunci API tidak boleh ditulis langsung di dalam kode (*hardcode*). Kita memuat API Key dari file `.env` lokal, dengan fallback ke Google Colab Secrets (jika suatu saat dijalankan di cloud).


In [ ]:
# 1. Coba baca dari file .env lokal (di root folder proyek)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
env_path = os.path.join(project_root, '.env')

if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path)
    print(f"📁 File .env ditemukan di: {env_path}")
else:
    # Coba load dari direktori saat ini
    load_dotenv()

api_key = os.getenv('GEMINI_API_KEY')

# 2. Fallback untuk Google Colab jika dijalankan di cloud
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
        print("🔑 Menggunakan API Key dari Colab Secrets.")
    except Exception:
        pass

if not api_key:
    raise ValueError("❌ GEMINI_API_KEY tidak ditemukan! Pastikan Anda sudah membuat file .env yang berisi GEMINI_API_KEY=your_key_here")

print("🔑 Gemini API Key berhasil dimuat!")
genai.configure(api_key=api_key)


## 3. Inisialisasi Model Gemini

Kita dapat memilih model Google Gemini yang memiliki penalaran konteks cepat dan akurat, misalnya `gemini-1.5-flash` atau `gemini-2.5-flash`.


In [ ]:
MODEL_NAME = "gemini-1.5-flash"

print(f"Menggunakan model: {MODEL_NAME}")
model = genai.GenerativeModel(model_name=MODEL_NAME)


## 4. Persona & System Prompt ELYSIA

`System Prompt` menentukan identitas, batasan, gaya bahasa, dan tujuan dari chatbot. 
ELYSIA dirancang dengan persona:
* Ramah, asyik, dan menggunakan istilah gaming secara natural dalam Bahasa Indonesia.
* Mampu mendeteksi **suasana hati (mood)**, **gaya bermain (playstyle DNA)**, dan **anggaran (budget Rupiah)**.
* Memberikan rekomendasi yang berorientasi pada kepuasan bermain pengguna.


In [ ]:
SYSTEM_PROMPT = """Kamu adalah ELYSIA (Emotionally-Adjusted Ludic Yield Spatial Integrated Assistant), asisten rekomendasi video game cerdas.

Persona & Gaya Bicara:
1. Ramah, asyik, antusias seputar game, menggunakan Bahasa Indonesia yang santai tapi sopan.
2. Kamu sangat mengerti nuansa gaming: gameplay loop, art style, mekanik santai vs kompetitif, dan ekosistem Steam.
3. Saat pengguna menyapa atau mengobrol, sambut mereka dengan hangat dan tanyakan game seperti apa yang sedang mereka cari, mood bermain saat ini, atau rentang budget mereka.
4. Jangan langsung memaksakan rekomendasi jika pengguna belum memberikan preferensi yang jelas; ajak berdiskusi santai terlebih dahulu.
5. Bila pengguna menyebutkan nominal budget, ingat bahwa dataset kita menggunakan mata uang Rupiah (IDR).
"""

# Konfigurasi model dengan System Prompt
elysia_model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(
        temperature=0.7,
        top_p=0.9
    )
)

print("✅ Model ELYSIA dengan System Prompt siap digunakan!")


## 5. Manajemen Conversation History (Multi-turn Chat)

LLM pada dasarnya stateless. Dengan memanfaatkan `start_chat(history=[])`, SDK Gemini secara otomatis mengelola memori giliran pesan (`user` dan `model`) sehingga ELYSIA selalu mengingat apa yang sudah dibicarakan sebelumnya.


In [ ]:
# Memulai sesi percakapan baru
chat_session = elysia_model.start_chat(history=[])

# Uji coba percakapan pertama
response1 = chat_session.send_message("Halo Elysia! Aku lagi cari rekomendasi game baru nih.")
print("ELYSIA:", response1.text)
print("-" * 60)

# Uji coba percakapan kedua (menguji apakah ELYSIA ingat konteks)
response2 = chat_session.send_message("Aku lebih suka yang santai dan visualnya adem. Kira-kira apa ya?")
print("ELYSIA:", response2.text)


## 6. Streaming Response (Menampilkan Jawaban Bertahap)

Untuk memberikan pengalaman interaktif (*real-time typing* seperti ChatGPT), kita dapat mengalirkan teks jawaban per *chunk* dengan `stream=True`.


In [ ]:
def kirim_pesan_streaming(session, user_message):
    """Mengirim pesan ke sesi chat dan mencetak jawaban secara bertahap (streaming)."""
    try:
        response = session.send_message(user_message, stream=True)
        print("ELYSIA : ", end="", flush=True)
        full_text = ""
        for chunk in response:
            if chunk.text:
                print(chunk.text, end="", flush=True)
                full_text += chunk.text
        print("\n")
        return full_text
    except Exception as e:
        print(f"\n⚠️ Terjadi kesalahan API: {e}")
        return None

# Contoh uji coba streaming
print("--- Demo Streaming Response ---")
_ = kirim_pesan_streaming(chat_session, "Aku punya budget sekitar 100 ribu di Steam, apakah cukup?")


## 7. Ekstraksi Preferensi Pengguna Berbasis Structured Prompting

Salah satu kemampuan utama ELYSIA adalah mengekstraksi **mood**, **dimensi Playstyle DNA**, **genre**, dan **maksimal budget IDR** dari ucapan bebas pengguna ke dalam format JSON terstruktur.

Dimensi Playstyle DNA:
1. `hardcore` (0.0 Casual – 1.0 Hardcore)
2. `complex` (0.0 Simple – 1.0 Complex)
3. `adrenaline` (0.0 Calming – 1.0 Adrenaline)


In [ ]:
EXTRACTION_PROMPT = """Tugasmu adalah menganalisis pesan pengguna dan mengekstrak preferensi gaming mereka ke dalam format JSON murni tanpa markdown atau pembungkus backticks.

Skema JSON yang diharapkan:
{
  "mood": "relaxed" | "competitive" | "immersive" | "focused" | null,
  "pref_genres": ["genre1", "genre2"],
  "max_budget": angka integer dalam Rupiah (IDR) atau null jika tidak disebutkan,
  "dna_estimate": {
    "hardcore": float antara 0.0 (sangat santai/casual) sampai 1.0 (sangat hardcore/tryhard),
    "complex": float antara 0.0 (mekanik simpel) sampai 1.0 (sistem mendalam/kompleks),
    "adrenaline": float antara 0.0 (sangat damai/calming) sampai 1.0 (penuh aksi/adrenalin)
  }
}

Contoh input: "Lagi capek habis ujian, pengen game santai pemandangan alam, budget maksimal 150rb"
Contoh output:
{
  "mood": "relaxed",
  "pref_genres": ["adventure", "casual", "indie"],
  "max_budget": 150000,
  "dna_estimate": {"hardcore": 0.2, "complex": 0.3, "adrenaline": 0.1}
}
"""

extractor_model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=EXTRACTION_PROMPT,
    generation_config=genai.GenerationConfig(
        temperature=0.1,
        response_mime_type="application/json"
    )
)

def ekstrak_preferensi(user_text):
    """Mengekstrak preferensi pengguna dari teks percakapan."""
    try:
        res = extractor_model.generate_content(f"Ekstrak preferensi dari teks ini: '{user_text}'")
        data = json.loads(res.text)
        return data
    except Exception as e:
        print(f"⚠️ Gagal mengekstrak preferensi: {e}")
        return {
            "mood": "relaxed",
            "pref_genres": [],
            "max_budget": None,
            "dna_estimate": {"hardcore": 0.5, "complex": 0.5, "adrenaline": 0.5}
        }

# Uji coba ekstraksi
sample_input = "Lagi suntuk pengen game shooter kompetitif yang menantang banget, uang jajan ada 200 ribu."
extracted = ekstrak_preferensi(sample_input)
print("Input User :", sample_input)
print("Hasil Ekstraksi:")
print(json.dumps(extracted, indent=2))


## 8. Memuat Dataset Video Game (`games.csv`)

Kita memuat dataset 24.082 video game yang sudah dilengkapi dengan metadata genre, platform, rating, metacritic, dan harga diskon Steam Indonesia (IDR).


In [ ]:
# Cek lokasi file dataset
dataset_candidates = [
    os.path.join(project_root, 'data', 'games.csv'),
    os.path.join(os.getcwd(), 'data', 'games.csv'),
    os.path.join('..', 'data', 'games.csv'),
    'data/games.csv'
]

dataset_path = None
for path in dataset_candidates:
    if os.path.exists(path):
        dataset_path = path
        break

if not dataset_path:
    raise FileNotFoundError("❌ Dataset games.csv tidak ditemukan di folder data/")

print(f"📁 Memuat dataset dari: {dataset_path}")
games_df = pd.read_csv(dataset_path)

# Pembersihan tipe data & penanganan nilai null
games_df['price_idr'] = pd.to_numeric(games_df['price_idr'], errors='coerce').fillna(0.0)
games_df['original_price_idr'] = pd.to_numeric(games_df['original_price_idr'], errors='coerce').fillna(0.0)
games_df['discount_percent'] = pd.to_numeric(games_df['discount_percent'], errors='coerce').fillna(0.0)
games_df['rating'] = pd.to_numeric(games_df['rating'], errors='coerce').fillna(0.0)
games_df['ratings_count'] = pd.to_numeric(games_df['ratings_count'], errors='coerce').fillna(0)
games_df['metacritic'] = pd.to_numeric(games_df['metacritic'], errors='coerce').fillna(0.0)
games_df['genres'] = games_df['genres'].fillna('')
games_df['platforms'] = games_df['platforms'].fillna('Unknown')
games_df['background_image'] = games_df['background_image'].fillna('')

print(f"📊 Total game berhasil dimuat: {len(games_df):,} baris")
display(games_df[['name', 'genres', 'rating', 'metacritic', 'price_idr', 'discount_percent']].head())


## 9. Prekomputasi Playstyle DNA Game

Setiap game dipetakan ke dalam ruang koordinat 3D $\in [0.0, 1.0]^3$:
1. **Casual vs Hardcore** (`dna_hardcore`): Didasarkan pada genre (RPG, Strategi vs Casual, Puzzle) serta popularitas dan metacritic.
2. **Simple vs Complex** (`dna_complex`): Didasarkan pada kedalaman sistem permainan.
3. **Calming vs Adrenaline** (`dna_adrenaline`): Didasarkan pada intensitas aksi.


In [ ]:
def precompute_game_dna(row):
    genres = [g.strip().lower() for g in str(row['genres']).split(',') if g.strip()]
    
    # 1. Casual vs Hardcore (0.0 = Casual, 1.0 = Hardcore)
    ch_base = 0.5
    hardcore_genres = {'rpg', 'strategy', 'shooter', 'simulation', 'massively multiplayer'}
    casual_genres = {'casual', 'puzzle', 'arcade', 'educational', 'card', 'board games', 'family'}
    
    for g in genres:
        if g in hardcore_genres:
            ch_base += 0.15
        elif g in casual_genres:
            ch_base -= 0.15
            
    if row['ratings_count'] > 1000:
        ch_base += 0.05
    if row['metacritic'] > 80:
        ch_base += 0.05
    ch = float(np.clip(ch_base, 0.0, 1.0))
    
    # 2. Simple vs Complex (0.0 = Simple, 1.0 = Complex)
    sc_base = 0.5
    complex_genres = {'strategy', 'rpg', 'simulation', 'massively multiplayer'}
    simple_genres = {'arcade', 'action', 'platformer', 'casual', 'puzzle', 'racing'}
    
    for g in genres:
        if g in complex_genres:
            sc_base += 0.20
        elif g in simple_genres:
            sc_base -= 0.10
    sc = float(np.clip(sc_base, 0.0, 1.0))
    
    # 3. Calming vs Adrenaline (0.0 = Calming, 1.0 = Adrenaline)
    ca_base = 0.5
    adrenaline_genres = {'shooter', 'action', 'fighting', 'racing', 'sports'}
    calming_genres = {'casual', 'puzzle', 'simulation', 'adventure', 'family'}
    
    for g in genres:
        if g in adrenaline_genres:
            ca_base += 0.20
        elif g in calming_genres:
            ca_base -= 0.15
    ca = float(np.clip(ca_base, 0.0, 1.0))
    
    return pd.Series([ch, sc, ca])

print("⏳ Menghitung Playstyle DNA untuk seluruh game...")
games_df[['dna_hardcore', 'dna_complex', 'dna_adrenaline']] = games_df.apply(precompute_game_dna, axis=1)
print("✅ Prekomputasi DNA selesai!")

display(games_df[['name', 'dna_hardcore', 'dna_complex', 'dna_adrenaline']].head())


## 10. Algoritma Hybrid Recommendation

Sistem menghitung skor rekomendasi multi-aspek:
1. **Mood Modifier**: Menyesuaikan koordinat DNA user berdasarkan mood sesaat (*relaxed, competitive, immersive, focused*).
2. **Skor DNA**: $1.0 - \frac{\text{Jarak Euclidean 3D}}{\sqrt{3}}$
3. **Skor Genre**: Overlap kesesuaian genre pilihan.
4. **Skor Rating**: Normalisasi gabungan Metacritic (60%) dan RAWG rating (40%).
5. **Skor Harga & Diskon**: Evaluasi apakah harga berada dalam batas anggaran dan pemberian insentif untuk diskon Steam aktif.


In [ ]:
def apply_mood_modifier(dna, mood):
    """Memodifikasi vektor DNA dasar pengguna berdasarkan pilihan mood saat ini."""
    adjusted = dna.copy()
    mood = (mood or '').lower()
    
    if mood == 'relaxed':
        adjusted['hardcore'] = max(0.0, adjusted['hardcore'] - 0.25)
        adjusted['complex'] = max(0.0, adjusted['complex'] - 0.20)
        adjusted['adrenaline'] = max(0.0, adjusted['adrenaline'] - 0.30)
    elif mood == 'competitive':
        adjusted['hardcore'] = min(1.0, adjusted['hardcore'] + 0.30)
        adjusted['complex'] = min(1.0, adjusted['complex'] + 0.15)
        adjusted['adrenaline'] = min(1.0, adjusted['adrenaline'] + 0.30)
    elif mood == 'immersive':
        adjusted['hardcore'] = min(1.0, adjusted['hardcore'] + 0.10)
        adjusted['complex'] = min(1.0, adjusted['complex'] + 0.25)
        adjusted['adrenaline'] = max(0.0, adjusted['adrenaline'] - 0.10)
    elif mood == 'focused':
        adjusted['hardcore'] = min(1.0, adjusted['hardcore'] + 0.20)
        adjusted['complex'] = min(1.0, adjusted['complex'] + 0.20)
        adjusted['adrenaline'] = max(0.0, adjusted['adrenaline'] - 0.05)
        
    return adjusted

def cari_rekomendasi_game(user_pref, top_n=5):
    """Menghasilkan daftar game terbaik berdasarkan preferensi & DNA pengguna."""
    df = games_df.copy()
    
    pref_genres = [g.lower() for g in user_pref.get('pref_genres', [])]
    mood = user_pref.get('mood')
    max_budget = user_pref.get('max_budget')
    raw_dna = user_pref.get('dna_estimate', {'hardcore': 0.5, 'complex': 0.5, 'adrenaline': 0.5})
    
    # 1. Sesuaikan DNA dengan Mood
    adj_dna = apply_mood_modifier(raw_dna, mood)
    
    # 2. Hitung Genre Match Score
    if pref_genres:
        def match_genre(g_str):
            g_list = [x.strip().lower() for x in str(g_str).split(',') if x.strip()]
            matches = sum(1 for x in g_list if any(p in x for p in pref_genres))
            return min(1.0, matches / len(pref_genres))
        df['score_genre'] = df['genres'].apply(match_genre)
    else:
        df['score_genre'] = 0.5 # Netral jika tidak menyebutkan genre
        
    # 3. Hitung Playstyle DNA Score (Euclidean Distance)
    game_dna_matrix = df[['dna_hardcore', 'dna_complex', 'dna_adrenaline']].values
    user_dna_vec = np.array([adj_dna['hardcore'], adj_dna['complex'], adj_dna['adrenaline']])
    distances = np.linalg.norm(game_dna_matrix - user_dna_vec, axis=1)
    df['score_dna'] = np.clip(1.0 - (distances / np.sqrt(3)), 0.0, 1.0)
    
    # 4. Hitung Rating Score
    meta_norm = df['metacritic'] / 100.0
    rawg_norm = df['rating'] / 5.0
    df['score_rating'] = np.where(df['metacritic'] > 0, (meta_norm * 0.6) + (rawg_norm * 0.4), rawg_norm)
    
    # 5. Hitung Price Score
    if max_budget is not None and max_budget > 0:
        def score_price(row):
            price = row['price_idr']
            if price == 0:
                return 1.0
            if price <= max_budget:
                bonus = (row['discount_percent'] / 100.0) * 0.10
                return min(1.0, 0.90 + bonus)
            else:
                penalty = (price - max_budget) / max_budget
                return max(0.0, 1.0 - penalty)
        df['score_price'] = df.apply(score_price, axis=1)
    else:
        df['score_price'] = 0.7 # Netral jika budget tidak dibatasi
        
    # 6. Total Weighted Match Score
    w_genre = 0.30 if pref_genres else 0.10
    w_dna = 0.40
    w_rating = 0.20
    w_price = 0.10 if max_budget else 0.05
    total_w = w_genre + w_dna + w_rating + w_price
    
    df['match_score'] = (
        (df['score_genre'] * w_genre) +
        (df['score_dna'] * w_dna) +
        (df['score_rating'] * w_rating) +
        (df['score_price'] * w_price)
    ) / total_w
    
    top_results = df.sort_values(by='match_score', ascending=False).head(top_n)
    return top_results, adj_dna

print("✅ Fungsi rekomendasi siap digunakan!")


## 11. Uji Coba Rekomendasi Berdasarkan Ekstraksi Teks

Mari kita uji alur rekomendasi dari teks percakapan pengguna ke hasil game terbaik.


In [ ]:
test_user_query = "Aku lagi pengen game petualangan santai eksplorasi alam, budget di bawah 100 ribu di Steam."
pref = ekstrak_preferensi(test_user_query)
results, final_dna = cari_rekomendasi_game(pref, top_n=5)

print(f"Hasil Ekstraksi Mood: {pref.get('mood')}")
print(f"Adjusted DNA: Hardcore={final_dna['hardcore']:.2f}, Complex={final_dna['complex']:.2f}, Adrenaline={final_dna['adrenaline']:.2f}\n")
print("Top 5 Rekomendasi Game:")
for idx, (_, row) in enumerate(results.iterrows(), 1):
    disc_text = f" (Diskon {int(row['discount_percent'])}%)" if row['discount_percent'] > 0 else ""
    price_text = f"Rp {int(row['price_idr']):,}" if row['price_idr'] > 0 else "Free to Play"
    print(f"{idx}. {row['name']} | Skor Kecocokan: {row['match_score']*100:.1f}%")
    print(f"   Genre: {row['genres']} | Harga: {price_text}{disc_text}")
    print(f"   Rating: {row['rating']}/5 | Metacritic: {int(row['metacritic'])}")
    print()


## 12. Menyimpan & Memuat Riwayat Percakapan (JSON)

Fitur ini berguna untuk merekam sesi percakapan ke dalam berkas JSON dan memuatnya kembali.


In [ ]:
def simpan_riwayat_chat(history, filename=None):
    """Menyimpan riwayat percakapan ke file JSON terstruktur."""
    if filename is None:
        filename = f"riwayat_chat_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
    serialized = []
    for msg in history:
        # Menangani objek Content dari Gemini SDK maupun dict standar
        if hasattr(msg, 'role') and hasattr(msg, 'parts'):
            text_parts = [p.text for p in msg.parts if hasattr(p, 'text')]
            serialized.append({
                "role": msg.role,
                "content": " ".join(text_parts)
            })
        elif isinstance(msg, dict):
            serialized.append(msg)
            
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(serialized, f, ensure_ascii=False, indent=2)
        
    print(f"💾 Riwayat percakapan berhasil disimpan ke: {filename}")
    return filename

def muat_riwayat_chat(filename):
    """Memuat kembali riwayat percakapan dari file JSON."""
    if not os.path.exists(filename):
        print(f"⚠️ File {filename} tidak ditemukan!")
        return []
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"📂 Berhasil memuat {len(data)} pesan dari {filename}")
    return data


## 13. Loop Chatbot Interaktif ELYSIA (CLI di Notebook)

Sekarang kita satukan seluruh pipeline menjadi asisten interaktif yang dapat diajak bercakap-cakap langsung di notebook.

Perintah khusus yang tersedia:
* `exit` : Keluar dari sesi percakapan.
* `clear` : Menghapus memori riwayat percakapan (memulai sesi baru).
* `save` : Menyimpan sesi obrolan saat ini ke file JSON.
* `recommend` : Memicu ELYSIA untuk mencari rekomendasi game terbaik berdasarkan konteks obrolan sejauh ini.


In [ ]:
def jalankan_elysia_interactive():
    print("=" * 65)
    print("       ELYSIA — Video Game Recommendation Assistant")
    print("=" * 65)
    print("Perintah khusus:")
    print("  'exit'      -> Keluar dari chatbot")
    print("  'clear'     -> Reset percakapan")
    print("  'save'      -> Simpan riwayat obrolan ke berkas JSON")
    print("  'recommend' -> Jalankan rekomendasi game berdasarkan obrolan")
    print("=" * 65 + "\n")
    
    chat = elysia_model.start_chat(history=[])
    full_conversation_log = []
    
    while True:
        try:
            user_input = input("Anda: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\nSampai jumpa lagi!")
            break
            
        if not user_input:
            continue
            
        cmd = user_input.lower()
        if cmd == 'exit':
            print("\nELYSIA: Terima kasih sudah mengobrol! Sampai jumpa di petualangan gaming berikutnya! 🎮")
            break
            
        if cmd == 'clear':
            chat = elysia_model.start_chat(history=[])
            full_conversation_log = []
            print("\n[Sistem] Konteks percakapan berhasil direset.\n")
            continue
            
        if cmd == 'save':
            simpan_riwayat_chat(full_conversation_log)
            print()
            continue
            
        if cmd == 'recommend':
            print("\n[Sistem] Menganalisis preferensi dari riwayat percakapan...")
            combined_context = " ".join([m['content'] for m in full_conversation_log if m['role'] == 'user'])
            if not combined_context:
                combined_context = "Game populer dan berkualitas tinggi."
            pref = ekstrak_preferensi(combined_context)
            top_games, dna = cari_rekomendasi_game(pref, top_n=5)
            
            # Buat teks ringkasan untuk diberikan ke ELYSIA agar disampaikan secara komunikatif
            games_summary = ""
            for idx, (_, row) in enumerate(top_games.iterrows(), 1):
                disc = f"(Diskon {int(row['discount_percent'])}%)" if row['discount_percent'] > 0 else ""
                price = f"Rp {int(row['price_idr']):,}" if row['price_idr'] > 0 else "Free to Play"
                games_summary += f"{idx}. {row['name']} (Match: {row['match_score']*100:.1f}%, Harga: {price} {disc}, Genre: {row['genres']})\n"
                
            prompt_rek = f"Sampaikan hasil rekomendasi game berikut kepada pengguna dengan gaya ramah ELYSIA:\n\n{games_summary}"
            answer = kirim_pesan_streaming(chat, prompt_rek)
            if answer:
                full_conversation_log.append({"role": "user", "content": "[Minta Rekomendasi Game]"})
                full_conversation_log.append({"role": "model", "content": answer})
            continue
            
        # Percakapan normal
        full_conversation_log.append({"role": "user", "content": user_input})
        answer = kirim_pesan_streaming(chat, user_input)
        if answer:
            full_conversation_log.append({"role": "model", "content": answer})

# Jalankan simulasi interaktif (uncomment baris di bawah untuk mencoba langsung di notebook)
# jalankan_elysia_interactive()


## 14. Kesimpulan & Ringkasan Eksperimen

Pada notebook ini, kita telah berhasil:
1. Menghubungkan Google Gemini API secara aman menggunakan `.env` lokal.
2. Menerapkan **System Prompt** dengan persona ELYSIA yang ramah dan memahami gaming.
3. Membangun mekanisme **Conversation History** dan **Streaming Response**.
4. Mengekstrak parameter preferensi, mood, dan koordinat Playstyle DNA pengguna secara otomatis.
5. Menghubungkan data ekstraksi dengan dataset 24.082 game dan menghitung rekomendasi multi-aspek (DNA Euclidean, Genre Overlap, Metacritic/RAWG, dan Harga Rupiah Steam).
6. Mengembangkan fitur **Save & Load** sesi percakapan.

Kode modular yang telah divalidasi pada notebook ini siap diintegrasikan ke `backend/recommendation_algorithm.py` dan `backend/chatbot.py` untuk antarmuka web interaktif!
